# Week 5 Laboratory
# Task 2: Image Classification using Convolutional Neural Networks (CNNs)

---

## Learning Objectives

After completing this laboratory, you should be able to:

✓ Explain why CNNs outperform MLPs on image data (local connectivity, parameter sharing, translation invariance).

✓ Understand key CNN layers: Conv2D, MaxPooling2D, Dropout, and Dense.

✓ Normalize raw image pixel arrays directly without manual feature extraction.

✓ Apply real-time data augmentation using ImageDataGenerator.

✓ Build, compile, and train a deep CNN in Keras with ReduceLROnPlateau scheduling.

✓ Evaluate performance using Accuracy, Precision, Recall, F1-score, and Confusion Matrix.

✓ Benchmark inference latency and perform qualitative error analysis.

---
## Setup Environment

Install required packages to run this notebook.

In [ ]:
!pip install opencv-python numpy matplotlib tqdm scikit-learn pandas tensorflow

---
## 1. Imports & Pre-defined Helper Functions

We import required libraries and provide utility functions for preprocessing, dataset loading, and visualization.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from timeit import default_timer as timer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Dense,
    Dropout,
    Flatten,
    Activation
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import warnings
warnings.filterwarnings('ignore')

LABELS = ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']

def preprocess_image(path_to_image, img_size=256):
    """Read and resize an input image."""
    img = cv2.imread(path_to_image, cv2.IMREAD_COLOR)
    img = cv2.resize(img, (img_size, img_size))
    return np.array(img)

def load_dataset(base_path='flowers', img_size=256):
    """Load dataset images and labels with automatic directory fallback."""
    if not os.path.exists(base_path):
        if os.path.exists(os.path.join('..', 'Week3', 'flowers')):
            base_path = os.path.join('..', 'Week3', 'flowers')
        elif os.path.exists(os.path.join('Week3', 'flowers')):
            base_path = os.path.join('Week3', 'flowers')
            
    X, Y = [], []
    for i in range(len(LABELS)):
        current_size = len(X)
        dir_path = os.path.join(base_path, LABELS[i])
        if os.path.exists(dir_path):
            for img in tqdm(os.listdir(dir_path), desc=f"Loading {LABELS[i]}"):
                if not img.startswith('.'):
                    X.append(preprocess_image(os.path.join(dir_path, img), img_size=img_size))
                    Y.append(LABELS[i])
            print(f'Loaded {len(X) - current_size} {LABELS[i]} images')
    return X, Y

def show_images(ground_truth, predictions, img_test, correct=True, num_images=5):
    """Show correctly or incorrectly classified test images in subplots."""
    count = 0
    plt.figure(figsize=(15, 3))
    for i in range(len(ground_truth)):
        if (ground_truth[i] == predictions[i]) == correct:
            count += 1
            plt.subplot(1, num_images, count)
            img = img_test[i]
            if img.max() <= 1.0:
                img = (img * 255).astype(np.uint8)
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            color = 'darkgreen' if correct else 'crimson'
            gt_name = LABELS[ground_truth[i]] if isinstance(ground_truth[i], (int, np.integer)) else ground_truth[i]
            pred_name = LABELS[predictions[i]] if isinstance(predictions[i], (int, np.integer)) else predictions[i]
            plt.title(f"GT: {gt_name}\nPred: {pred_name}", fontsize=9, color=color, fontweight='bold')
            plt.axis('off')
            if count == num_images:
                break
    plt.tight_layout()
    plt.show()

print("Environment set up successfully!")

---
## STEP 1. Load Dataset

Load raw flower images ($256 \times 256 \times 3$) using load_dataset(img_size=256).

In [ ]:
# STEP 1. Load dataset
# UNCOMMENT BELOW
# X, Y = load_dataset('flowers', img_size=256)

print(f"Total samples loaded: {len(X)}")

---
## STEP 2. Normalize Pixel Values

Scale raw pixel values from $[0, 255]$ to floating-point $[0.0, 1.0]$ for stable neural network training.

In [ ]:
# STEP 2. Normalize pixel values
# Convert X to float32 NumPy array and scale pixel values to [0.0, 1.0]
# YOUR CODE HERE
# X = np.array(..., dtype=np.float32) / ...

print(f"X shape: {X.shape}, min: {X.min()}, max: {X.max()}")

---
## STEP 3. Encode Labels to One-Hot Vectors

Convert categorical string labels into 5-dimensional one-hot vectors using LabelEncoder and to_categorical.

In [ ]:
# STEP 3. Encode labels to one-hot encoding
# Transform string labels in Y to integer IDs, then convert to 5-class one-hot vectors
# YOUR CODE HERE
# label_encoder = LabelEncoder()
# Y_int = label_encoder.fit_transform(...)
# Y = to_categorical(..., num_classes=5)

print(f"One-hot target shape: {Y.shape}")

---
## STEP 4. Split Dataset (Train, Validation, Test)

Split data into **Train (60%)**, **Validation (20%)**, and **Test (20%)** sets with random_state=1.

In [ ]:
# STEP 4. Split dataset into train, validation, and test sets
# 1. Split 20% validation set (img_val, label_val) from (X, Y)
# 2. Split 25% test set (img_test, label_test) from remaining training data
# YOUR CODE HERE
# img_train, img_val, label_train, label_val = train_test_split(..., ..., test_size=0.2, random_state=1)
# img_train, img_test, label_train, label_test = train_test_split(..., ..., test_size=0.25, random_state=1)

print(f"Training samples  : {len(img_train)} ({len(img_train)/len(X)*100:.1f}%)")
print(f"Validation samples: {len(img_val)} ({len(img_val)/len(X)*100:.1f}%)")
print(f"Test samples      : {len(img_test)} ({len(img_test)/len(X)*100:.1f}%)")

---
## STEP 5. Data Augmentation

Apply real-time data augmentation (rotations, zoom, shifts, horizontal flips) with ImageDataGenerator to improve generalization.

In [ ]:
# STEP 5. Create data generator with augmentation
# UNCOMMENT BELOW
# data_generator = ImageDataGenerator(
#     rotation_range=10,
#     zoom_range=0.1,
#     width_shift_range=0.2,
#     height_shift_range=0.2,
#     horizontal_flip=True,
#     vertical_flip=False
# )
# data_generator.fit(img_train)

print("Data generator configured successfully!")

---
## STEP 6. Build Deep CNN Architecture

Construct a deep CNN with 5 Conv-Pool blocks and a Dense classification head:
* Conv2D(32) $\rightarrow$ Conv2D(64) $\rightarrow$ Conv2D(128) $\rightarrow$ Conv2D(256) $\rightarrow$ Conv2D(512)
* Flatten $\rightarrow$ Dense(1024) $\rightarrow$ Dropout(0.5) $\rightarrow$ Dense(512) $\rightarrow$ Dense(5, 'softmax')

In [ ]:
# STEP 6. Create CNN model
def create_model(img_size=256):
    """Create deep 5-block CNN model."""
    model = Sequential()
    
    # Block 1: Conv2D(32, (5,5), padding='same', activation='relu', input_shape=(img_size, img_size, 3)) + MaxPooling2D((2,2))
    # Block 2: Conv2D(64, (3,3), padding='same', activation='relu') + MaxPooling2D((2,2))
    # Block 3: Conv2D(128, (3,3), padding='same', activation='relu') + MaxPooling2D((2,2))
    # Block 4: Conv2D(256, (3,3), padding='same', activation='relu') + MaxPooling2D((2,2))
    # Block 5: Conv2D(512, (3,3), padding='same', activation='relu') + MaxPooling2D((2,2))
    # Classification Head: Flatten() -> Dense(1024, 'relu') -> Dropout(0.5) -> Dense(512, 'relu') -> Dense(5, 'softmax')
    # YOUR CODE HERE
    # model.add(Conv2D(32, (5, 5), padding='same', activation='relu', input_shape=(img_size, img_size, 3)))
    # model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
    # model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
    # model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
    # model.add(Conv2D(128, (3, 3), padding=..., activation=...))
    # model.add(MaxPooling2D(pool_size=..., strides=...))
    # model.add(Conv2D(..., (3, 3), padding='same', activation='relu'))
    # model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
    # model.add(Conv2D(..., (3, 3), padding='same', activation='relu'))
    # model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
    # model.add(Flatten())
    # model.add(Dense(1024, activation='relu'))
    # model.add(Dropout(0.5))
    # model.add(Dense(..., activation=...))
    # model.add(Dense(..., activation='softmax'))
    
    return model

model = create_model(img_size=256)
model.summary()

---
## STEP 7. Compile the CNN Model

Compile the model with Adam(0.001) optimizer and categorical_crossentropy loss.

In [ ]:
# STEP 7. Define optimizer and loss function
# UNCOMMENT BELOW
# model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

print("Model compiled successfully!")

---
## STEP 8. Train the CNN Model

Train with model.fit() and ReduceLROnPlateau for dynamic learning rate scheduling.

In [ ]:
# STEP 8. Train the CNN model
lr_scheduler = ReduceLROnPlateau(monitor='val_accuracy', factor=0.1, patience=5, verbose=1, min_lr=1e-6)

# Train model using model.fit() for 30 epochs with batch_size=32, validation_data, and callbacks=[lr_scheduler]
# Record training duration using timer()
# YOUR CODE HERE
# start = timer()
# history = model.fit(
#     x=..., y=...,
#     epochs=30, batch_size=32,
#     validation_data=(..., ...),
#     callbacks=[lr_scheduler],
#     verbose=1
# )
# end = timer()
# print(f"Training completed in {end - start:.2f} seconds")

---
## STEP 9. Plot Loss and Accuracy Curves

Visualize training and validation Loss and Accuracy trajectories across epochs.

In [ ]:
# STEP 9. Plot Loss and Accuracy curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(history.history['accuracy'], label='Train Acc', lw=2)
ax1.plot(history.history['val_accuracy'], label='Val Acc', lw=2, linestyle='--')
ax1.set_title('CNN Accuracy vs. Epochs', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train Loss', lw=2)
ax2.plot(history.history['val_loss'], label='Val Loss', lw=2, linestyle='--')
ax2.set_title('CNN Loss vs. Epochs', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## STEP 10. Benchmark Inference Speed on Test Set

Evaluate test predictions and measure average inference latency per image.

In [ ]:
# STEP 10. Benchmark inference latency
# Measure time taken to compute model.predict(img_test, batch_size=32)
# YOUR CODE HERE
# start = timer()
# raw_predictions = model.predict(..., batch_size=32)
# end = timer()

latency = (end - start) / len(img_test)
print(f"Inference latency: {latency:.6f} s/image ({latency * 1000:.3f} ms/image)")

---
## STEP 11. Report Classification Metrics

Calculate Accuracy, Macro Precision, Recall, and F1-score on the test set.

In [ ]:
# STEP 11. Report classification metrics
# 1. Convert one-hot predictions to integer class IDs with np.argmax(..., axis=1)
# 2. Compute accuracy_score, precision_score, recall_score, and f1_score (average='macro')
# YOUR CODE HERE
# y_pred = np.argmax(raw_predictions, axis=1)
# y_test = np.argmax(label_test, axis=1)
# accuracy  = accuracy_score(..., ...)
# precision = precision_score(..., ..., average='macro')
# recall    = recall_score(..., ..., average='macro')
# f1        = f1_score(..., ..., average='macro')

print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}\n")
print(classification_report(y_test, y_pred, target_names=LABELS))

---
## STEP 12. Plot Confusion Matrix

Display the confusion matrix for CNN test predictions.

In [ ]:
# STEP 12. Plot confusion matrix
# UNCOMMENT BELOW
# ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=LABELS, cmap='Blues', xticks_rotation=45)
# plt.title('Confusion Matrix - Keras CNN', fontweight='bold')
# plt.show()

---
## STEP 13. Qualitative Error Analysis

Inspect top 5 correctly classified and top 5 misclassified test images.

In [ ]:
# STEP 13. Show correctly and incorrectly classified images
# UNCOMMENT BELOW
# show_images(y_test, y_pred, img_test, correct=True, num_images=5)
# show_images(y_test, y_pred, img_test, correct=False, num_images=5)

---
## Reflection Questions

1. **Q1:** Why do CNNs achieve higher accuracy than MLPs on images? What specific operations enable this?
2. **Q2:** What is the purpose of Max Pooling and how does it reduce computational cost?
3. **Q3:** Why is Dropout(0.5) placed before the classification head?
4. **Q4:** How does Data Augmentation prevent overfitting when training deep neural networks?